In [ ]:
import pandas as pd

df = pd.read_csv(
    "../data/raw/PS_20174392719_1491204439457_log.csv",
    nrows=1000
)

print(df.dtypes)
print(df.columns.tolist())

step                int64
type               object
amount            float64
nameOrig           object
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest           object
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


In [4]:
print(df)


print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))
print("Shape:", df.shape)

     step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0       1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1       1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2       1  TRANSFER    181.00  C1305486145          181.0            0.00   
3       1  CASH_OUT    181.00   C840083671          181.0            0.00   
4       1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   
..    ...       ...       ...          ...            ...             ...   
995     1   PAYMENT   2167.26   C732597634        13669.0        11501.74   
996     1   PAYMENT   9738.95  C2031927175       289748.0       280009.05   
997     1   PAYMENT   3396.25  C1967617997        18524.0        15127.75   
998     1   PAYMENT   6780.78   C925803196        52640.0        45859.22   
999     1   PAYMENT   2284.54    C49318987          539.0            0.00   

        nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  

## Check for invalid data

In [8]:
#Null and unique
null_counts = df.isna().sum()
null_percent = df.isna().mean() * 100
unique_counts = df.nunique(dropna=True)
print("number of Null: \n")
print(null_counts)
print("\n Number of unique: \n")
print(unique_counts)

number of Null: 

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

 Number of unique: 

step                 1
type                 5
amount             996
nameOrig          1000
oldbalanceOrg      674
newbalanceOrig     548
nameDest           597
oldbalanceDest     556
newbalanceDest     114
isFraud              2
isFlaggedFraud       1
dtype: int64


In [ ]:

#numeric ranges check for any number column
numeric_columns = df.select_dtypes(include="number").columns

numeric_stats = df[numeric_columns].agg(
    ["min", "max", "mean", "median", "std"]
).T

print(numeric_stats)

                 min          max          mean     median           std
step            1.00         1.00  1.000000e+00      1.000  0.000000e+00
amount          8.73   2545478.01  1.181995e+05  14658.085  2.484799e+05
oldbalanceOrg   0.00   9716462.14  8.646043e+05  14518.000  2.078040e+06
newbalanceOrig  0.00   9987286.56  8.849988e+05   5661.830  2.127790e+06
oldbalanceDest  0.00  17664818.40  6.657141e+05  28755.055  2.060515e+06
newbalanceDest  0.00  19169204.93  1.224672e+06      0.000  3.436780e+06
isFraud         0.00         1.00  9.000000e-03      0.000  9.448771e-02
isFlaggedFraud  0.00         0.00  0.000000e+00      0.000  0.000000e+00


In [10]:
#Duplicate
duplicate_rows = df.duplicated().sum()

origin_repeated = df["nameOrig"].duplicated().sum()
destination_repeated = df["nameDest"].duplicated().sum()

print("Duplicate rows:", duplicate_rows)
print("Repeated origin IDs:", origin_repeated)
print("Repeated destination IDs:", destination_repeated)

Duplicate rows: 0
Repeated origin IDs: 0
Repeated destination IDs: 403


In [11]:
# Export report csv
OUTPUT_PATH = "../reports/data_quality/profile.csv"

profile = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "null_count": null_counts.values,
    "null_percent": null_percent.values,
    "unique_count": unique_counts.values
})

profile["min"] = profile["column"].map(numeric_stats["min"])
profile["max"] = profile["column"].map(numeric_stats["max"])
profile["mean"] = profile["column"].map(numeric_stats["mean"])
profile["median"] = profile["column"].map(numeric_stats["median"])
profile["std"] = profile["column"].map(numeric_stats["std"])

profile.to_csv(OUTPUT_PATH, index=False)

print(f"Saved: {OUTPUT_PATH}")

Saved: ../reports/data_quality/profile.csv
